# 01. Exploratory Data Analysis (EDA)
**Project:** Stock Market Data Analyzer  
**Author:** Akash Das  
**Phase:** 5 — Exploratory Data Analysis  
**Status:** Validated Implementation

---

### Purpose and Architecture Scope
This notebook implements the structured Exploratory Data Analysis (EDA) workflow for the **Stock Market Data Analyzer** project.

The objective of Phase 5 is to profile cleaned historical market data, compute descriptive statistical summaries, evaluate intraday price spreads and trading volume distributions, and quantify Pearson correlation structures among numeric features.

> **Strict Phase 5 Boundaries**:
> - Descriptive statistical profiling only.
> - **NO** moving averages (Phase 6: SMA 20, SMA 50, EMA 20).
> - **NO** returns, volatility, or drawdown modeling (Phase 7).
> - **NO** interactive Plotly visualizations (Phase 8).
> - **NO** Streamlit dashboard UI (Phase 9).
> - **NO** predictive models or automated investment recommendations (Phases 10-12).


## 2. Imports and Configuration
We import necessary data manipulation, visualization, and project analysis modules. Matplotlib and Seaborn are configured for clear, reproducible static rendering.


In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root 'src/' directory is in sys.path
cwd = Path.cwd()
project_root = cwd if (cwd / "src").exists() else cwd.parent
src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Import stock_analyzer modules
from stock_analyzer.data.loader import load_sample_stock_data, load_stock_csv
from stock_analyzer.data.cleaner import clean_stock_data
from stock_analyzer.analysis.eda import (
    ExploratoryDataAnalyzer,
    generate_summary_statistics,
    compute_correlation_matrix,
    generate_eda_summary,
)

# Configuration
pd.set_option("display.max_columns", 10)
pd.set_option("display.precision", 2)
sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams["figure.autolayout"] = True

print(f"Project root resolved to: {project_root}")
print("Dependencies and analytical modules imported successfully.")


## 3. Loading a Cleaned Dataset
We load a processed, schema-validated stock dataset. If pre-cleaned real market data exists in `data/processed/`, it is utilized. Otherwise, we load the deterministic development sample from `data/sample/sample_stock.csv` and clean it via `clean_stock_data()`.

*Note on Synthetic Data: If the development sample is loaded, it is explicitly flagged and documented as synthetic.*


In [ ]:
processed_dir = project_root / "data" / "processed"
processed_files = list(processed_dir.glob("*.csv"))

# Filter out empty or gitkeep files
valid_processed = [f for f in processed_files if f.stat().st_size > 50]

if valid_processed:
    data_path = valid_processed[0]
    ticker_name = data_path.stem
    print(f"Loading processed dataset from: {data_path.name}")
    df_raw = load_stock_csv(data_path)
    clean_result = clean_stock_data(df_raw)
    df = clean_result.data
    df.attrs["ticker"] = ticker_name
    is_synthetic = bool(df.attrs.get("is_synthetic", False))
else:
    print("Loading deterministic development baseline: data/sample/sample_stock.csv [SYNTHETIC]")
    df_raw = load_sample_stock_data()
    clean_result = clean_stock_data(df_raw)
    df = clean_result.data
    df.attrs["ticker"] = "SAMPLE"
    is_synthetic = True

print("Data loaded successfully.")
print(f"Ticker:        {clean_result.data.attrs.get('ticker', 'UNKNOWN')}")
print(f"Synthetic:     {is_synthetic}")
print(f"Observations:  {len(df)}")
min_date_str = str(pd.to_datetime(df['Date']).min())[:10]
max_date_str = str(pd.to_datetime(df['Date']).max())[:10]
print(f"Date Range:    {min_date_str} to {max_date_str}")


## 4. Dataset Inspection
We inspect the schema structure, data types, and sample observations of the cleaned dataset.


In [ ]:
print("=== Dataset Info ===")
df.info()

print("\n=== First 5 Observations ===")
print(df.head())

print("\n=== Last 5 Observations ===")
print(df.tail())


## 5. Descriptive Statistics
Descriptive statistics summarize the central tendency, dispersion, and shape of numeric OHLCV variables.
We compute count, mean, standard deviation, minimum, quartiles (25%, 50%, 75%), maximum, Interquartile Range (IQR), and tail percentiles (5%, 95%).


In [ ]:
analyzer = ExploratoryDataAnalyzer(df)
summary_stats = analyzer.compute_summary_statistics(percentiles=[0.05, 0.95])

print("=== Comprehensive Descriptive Statistics ===")
print(summary_stats.to_string())


## 6. Price and Volume Exploration
Here we analyze intraday price dynamics, daily High-Low spreads, historical extremes, and the relationship between price levels and trading volume.


In [ ]:
pv_summary = analyzer.compute_price_and_volume_summary()

print("=== Price Extremes ===")
for k, v in pv_summary["price_extremes"].items():
    print(f"  {k:20s}: {v}")

print("\n=== Daily Spread Dynamics ===")
for k, v in pv_summary["daily_spread"].items():
    print(f"  {k:20s}: {v}")

print("\n=== Volume Summary ===")
for k, v in pv_summary["volume_summary"].items():
    print(f"  {k:20s}: {v}")

print("\n=== Price-Volume Relationship ===")
print(f"  Pearson Correlation: {pv_summary['price_volume_relationship']['pearson_correlation']:.4f}")
print(f"  Interpretation:      {pv_summary['price_volume_relationship']['interpretation']}")


## 7. Correlation Analysis
Correlation measures the strength and direction of linear association between pairs of numerical variables.

### Methodology & Limitations
- **Metric**: Pearson correlation coefficient ($r \in [-1, 1]$).
- **Assumptions**: Evaluates pairwise linear relationships; does not capture non-linear dependencies.
- **Symmetry**: $r(X, Y) = r(Y, X)$, with diagonal entries equal to $1.0$.
- **Causality Warning**: A strong correlation between OHLC variables reflects inherent time-series co-movement across daily price discovery, **NOT** a causal relationship. Correlation does not imply causation.


In [ ]:
corr_matrix = analyzer.compute_correlations(method="pearson")

print("=== Pearson Correlation Matrix ===")
print(corr_matrix.to_string())


## 8. Exploratory Visualizations
We generate static exploratory charts using Matplotlib and Seaborn, saving each high-resolution figure to `outputs/charts/`:
1. Closing Price Distribution (Histogram + KDE)
2. Trading Volume Distribution
3. OHLC Boxplot Distributions
4. Feature Correlation Heatmap
5. High-Low Price Spread Analysis (Time Series & Histogram)


In [ ]:
charts_dir = project_root / "outputs" / "charts"
charts_dir.mkdir(parents=True, exist_ok=True)
ticker_symbol = analyzer.ticker

# 8.1 Price Distribution
fig_price = analyzer.plot_price_distribution(
    column="Close",
    bins=25,
    output_path=charts_dir / f"{ticker_symbol}_price_distribution.png"
)
plt.show()

# 8.2 Volume Distribution
fig_vol = analyzer.plot_volume_distribution(
    bins=25,
    output_path=charts_dir / f"{ticker_symbol}_volume_distribution.png"
)
plt.show()

# 8.3 OHLC Boxplot Distributions
fig_ohlc = analyzer.plot_ohlc_distributions(
    output_path=charts_dir / f"{ticker_symbol}_ohlc_boxplot.png"
)
plt.show()

# 8.4 Correlation Heatmap
fig_corr = analyzer.plot_correlation_heatmap(
    method="pearson",
    output_path=charts_dir / f"{ticker_symbol}_correlation_heatmap.png"
)
plt.show()

# 8.5 High-Low Spread Analysis
fig_spread = analyzer.plot_high_low_spread(
    output_path=charts_dir / f"{ticker_symbol}_hl_spread.png"
)
plt.show()

print(f"All visualizations successfully generated and saved to: {charts_dir}")


## 9. Key Observations
From our exploratory analysis:
1. **Price Dispersion & Central Tendency**:
   - The median and mean close prices reflect the central level of the price distribution.
   - The Interquartile Range (IQR) captures 50% of the daily closing price variations.
2. **Intraday Volatility / Price Spread**:
   - The High - Low price spread illustrates daily trading ranges. Days with wide spreads reflect heightened intraday uncertainty.
3. **Volume Characteristics**:
   - Trading volume shows characteristic right-skewness common to equity markets, where typical trading days exhibit baseline volume punctuated by occasional high-volume activity.
4. **Correlation Structure**:
   - Price features (`Open`, `High`, `Low`, `Close`) display near-perfect positive correlation ($r > 0.98$), consistent with daily financial time series where all metrics track the same underlying asset value.
   - Price vs. Volume correlation provides insight into whether volume systematically expands during price advances or pullbacks.


## 10. Limitations and Conclusion
### Limitations of Phase 5 EDA
- **Descriptive Only**: EDA summarizes historical observations; it does not forecast future price action or evaluate investment profitability.
- **Stationarity**: Stock price series are inherently non-stationary; raw descriptive statistics are sensitive to the chosen observation window.
- **No Smoothing or Indicators**: Raw prices contain high-frequency market noise that requires moving averages (Phase 6) and return calculations (Phase 7) to model trends and risk.

### Transition to Phase 6
The verified outputs and descriptive baseline produced in Phase 5 will feed into **Phase 6: Moving Average Calculation**, where we implement Simple Moving Averages (SMA 20, SMA 50) and Exponential Moving Averages (EMA 20) for trend filtering.
